In [1]:
!pip install pandas scikit-learn matplotlib seaborn joblib



[notice] A new release of pip is available: 25.2 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import pandas as pd
import numpy as np
from sklearn.datasets import fetch_openml

In [3]:
# 1. Load Australian Credit Approval Dataset via OpenML (Bypasses raw URL HTTP errors)
credit = fetch_openml(name='credit-approval', version=1, as_frame=True)
df = credit.frame.dropna()

# Map the target column to binary integers (0 and 1) if it's categorical
if df['class'].dtype.name == 'category':
    df['class'] = df['class'].cat.codes
df = df.rename(columns={'class': 'target'})

print(f"Dataset Loaded Successfully!")
print(f"Total Features (excluding target): {df.shape[1]-1} (Requirement: Min 12)[cite: 3]")
print(f"Total Instances: {df.shape[0]} (Requirement: Min 500)[cite: 3]")

Dataset Loaded Successfully!
Total Features (excluding target): 15 (Requirement: Min 12)[cite: 3]
Total Instances: 653 (Requirement: Min 500)[cite: 3]


In [4]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score, roc_auc_score, precision_score, 
    recall_score, f1_score, matthews_corrcoef
)

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import SVC

from sklearn.calibration import CalibratedClassifierCV

In [5]:
# 2. Train-Test Split (80% train, 20% test)
train_df, test_df = train_test_split(df, test_size=0.2, random_state=42, stratify=df['target'])

In [6]:
train_df

,A1,A2,A3,A4,A5,A6,A7,A8,A9,A10,A11,A12,A13,A14,A15,target
627,b,36.08,2.540,u,g,ff,ff,0.000,f,f,0,f,g,0.0,1000,1
555,b,41.92,0.420,u,g,c,h,0.210,t,t,6,f,g,220.0,948,0
191,b,42.00,0.205,u,g,i,h,5.125,t,f,0,f,g,400.0,0,0
268,b,59.67,1.540,u,g,q,v,0.125,t,f,0,t,g,260.0,0,0
408,b,16.00,3.125,u,g,w,v,0.085,f,t,1,f,g,0.0,6,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
31,b,42.00,9.790,u,g,x,h,7.960,t,t,8,f,g,0.0,0,0
582,b,48.50,4.250,u,g,m,v,0.125,t,f,0,t,g,225.0,0,0
623,b,18.83,0.000,u,g,q,v,0.665,f,f,0,f,g,160.0,1,1
82,b,39.83,0.500,u,g,m,v,0.250,t,f,0,f,s,288.0,0,1


In [7]:
test_df

,A1,A2,A3,A4,A5,A6,A7,A8,A9,A10,A11,A12,A13,A14,A15,target
49,b,23.92,0.665,u,g,c,v,0.165,f,f,0,f,g,100.0,0,0
317,b,17.50,22.000,l,gg,ff,o,0.000,f,f,0,t,p,450.0,100000,0
209,b,39.50,4.250,u,g,c,bb,6.500,t,t,16,f,g,117.0,1210,0
514,b,22.67,2.540,y,p,c,h,2.585,t,f,0,f,g,0.0,0,0
1,a,58.67,4.460,u,g,q,h,3.040,t,t,6,f,g,43.0,560,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
581,b,41.00,0.040,u,g,e,v,0.040,f,t,1,f,s,560.0,0,0
249,b,21.83,11.000,u,g,x,v,0.290,t,t,6,f,g,121.0,0,0
525,b,45.17,1.500,u,g,c,v,2.500,t,f,0,t,g,140.0,0,1
48,b,41.50,1.540,u,g,i,bb,3.500,f,f,0,f,g,216.0,0,0


In [8]:
# Save test data as required by the assignment[cite: 6]
test_df.to_csv('test_data.csv', index=False)
print("test_data.csv saved successfully!")


test_data.csv saved successfully!


In [9]:
# 3. Feature Scaling (Crucial for Logistic Regression, KNN, and SVM)
X_train = train_df.drop(columns=['target'])
y_train = train_df['target']
X_test = test_df.drop(columns=['target'])
y_test = test_df['target']


In [11]:

# One-hot encode any remaining categorical columns automatically
X_train = pd.get_dummies(X_train, drop_first=True)
X_test = pd.get_dummies(X_test, drop_first=True)
X_train, X_test = X_train.align(X_test, join='left', axis=1, fill_value=0)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)



In [12]:
import os
import joblib
os.makedirs('model', exist_ok=True)
joblib.dump(scaler, 'model/scaler.joblib')

['model/scaler.joblib']